# Exercise 2.1.9 — Implement reward averaging

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `2.1 Intro to RL`  
**Notebook:** `2.1_Intro_to_RL_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=2.1.9](https://delta-drills.vercel.app/?arena_exercise=2.1.9)


# [2.1] - Intro to RL (exercises)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/01_[2.1]_Intro_to_RL)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part1_intro_to_rl/2.1_Intro_to_RL_exercises.ipynb?t=20260303) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part1_intro_to_rl/2.1_Intro_to_RL_solutions.ipynb?t=20260303)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-21.png" width="350">

# Introduction

This section is designed to bring you up to speed with the basics of reinforcement learning. Before we cover the big topics like DQN, VPG, PPO and RLHF, we need to build a strong theoretical foundation first.

For now, we will assume the environment has a finite set of states/actions, and we explore two cases:
1. that the dynamics of the environment are directly accessible to the agent, so that we can solve for an optimal policy *analytically*. 
2. the environment is a black box that we can sample from, and the agent must learn from play how the environment works. 

Tomorrow, we weaken the assumption that there are few enough states we could visit them all, and instead build something with deep learning that can generalize to never-before-seen states.

This day's material will be more theory heavy, as the goal is to understand the fundamentals of RL before we apply deep learning to it.

For a lecture on the material today, which provides some high-level understanding before you dive into the material, watch the video below:

<iframe width="540" height="304" src="https://www.youtube.com/embed/Q2cFcq3I0G8" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Content & Learning Objectives

### 1️⃣ Planning

> ##### Learning Objectives
> * Understand Markov Decision Processes and the goal of the agent, for known environments.
> * Understand the Bellman equation.
> * Understand the policy improvement theorem, and how we can use it to iteratively solve for an optimal policy.

### 2️⃣ Learning

* **SARSA** and **Q-learning** are methods of learning in Markovian domains by experiencing the consequences of actions, and learning from them, without peeking directly at how the environment works. The agent must learn the dynamic of the environment solely from experience.

* We can do short updates and bootstrap the rest, or we can do long updates, but suffer higher variance. Eligibility traces allow us to get the best of both worlds in an efficient fashion.

> ##### Learning Objectives
> * Understand the anatomy of a `gym.Env`, so that you feel comfortable using them and writing your own.
> * Understand the ideas behind temporal difference methods such as SARSA and Q-learning.
> * Implement SARSA and Q-Learning, and compare them on different environments.
> * Understand the TD($\lambda$) algorithm, and how it can we used to mix over short and long timescale updates.

### 3️⃣ (Optional) Multi-Armed Bandit

Multi-armed bandits are a different flavour of environment where there are no states, but the environmental distribution itself may drift over time. Here, the emphasis is on strategies to trade-off exploration v.s. exploitation, and methods from bandits (such as UCB) are used elsewhere 
(e.g [Monte Carlo Tree Search](https://en.wikipedia.org/wiki/Monte_Carlo_tree_search), or the [Hyperband algorithm](https://arxiv.org/abs/1603.06560) for hyperparameter search). 

This used to be core material, but as none of the later material requires understanding bandits, we've moved it to bonus optional material.

> ##### Learning Objectives
> * Understand the difficulty of optimal exploration
> * Compare *Upper Confidence Bound* (UCB) methods to other exploration methods.

## Setup code

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter2_rl"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import jaxtyping
except:
    %pip install wandb==0.18.7 einops "gymnasium[atari, accept-rom-license, other]==0.29.0" pygame jaxtyping

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import TypeAlias

import einops
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

Arr: TypeAlias = np.ndarray

max_episode_steps = 1000
N_RUNS = 200

# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part1_intro_to_rl"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part1_intro_to_rl.tests as tests
import part1_intro_to_rl.utils as utils
from plotly_utils import cliffwalk_imshow, imshow, line

# 1️⃣ Planning

> ##### Learning Objectives
>
> * Understand the Bellman equation.
> * Understand the policy improvement theorem, and how we can use it to iteratively solve for an optimal policy.

## Planning in Known Environments

We are presented with a white-box environment: we have access to the environmental distribution $T$ and reward function $R$. The goal is to turn the problem of finding the best policy $\pi^*$ into an optimization problem, which we can then solve iteratively.

Here, we assume environments small enough that visiting all state-action pairs is tractable. Tabular RL doesn't learn any relationships between states that could be treated similarly, but instead keep track of how valuable each state is in a large lookup table.

## Readings (optional)

There are no compulsory readings before the material in this section (the lecture covers everything), although the following sections of [Sutton and Barto](https://www.andrew.cmu.edu/course/10-703/textbook/BartoSutton.pdf) are relevant and so you may want to refer to them if you get stuck / confused with anything:

- Chapter 3: Sections 3.1, 3.2, 3.3, 3.4, 3.5, 3.6
- Chapter 4: Sections 4.1, 4.2, 4.3, 4.4

## What is Reinforcement Learning?

In reinforcement learning, the agent interacts with an environment in a loop: The agent in state $s$ uses its current policy $\pi$ to sample an **action** $a \sim \pi(\cdot | s)$ to the environment, and the environment replies with **state, reward** pair $(s',r)$, where the new state is sampled from the enviromental transition distribution $s' \sim T(\cdot | s,a)$ conditioned on the previous state $s$ and action $a$, and the reward $r = R(s,a,s')$ is computed using the reward function $R$, given the state,action,next-state triple $(s,a,s')$. This implies both the environment and the policy are **Markovian**, in that their dynamics depend only on the previous state and action (for the environment) or on the current state (for the policy).

**Note** - some of the sections below have a TLDR at the start. This indicates that not the entire section is crucially important to read and understand (e.g. it's a small pedantic note or a mathematical divergence), and you can safely read the TLDR and move onto the next section if you want.

<details>
<summary>Policy vs Agent?</summary>

Technically, the agent is the decision process that chooses the current policy, and the policy itself is just a conditional distribution over actions given the current state.
We're often sloppy and use the terms interchangeably (e.g. "the agent chooses an action"), but it's important to be aware of the distinction.
</details>

### Trajectories

We define a **trajectory** or **rollout** (denoted $\tau$) in RL as the full history of interaction between the agent and the environment:

$$
\tau = s_0, a_0, r_1, s_1, a_1, r_2, s_2, a_2, r_3, \ldots
$$

Under this notation, the timestep increments during the environment's turn to interact. In other words, during timestep $t$, given state $s_t$ the agent returns action $a_t$ (in the same time step), but given $(s_t, a_t)$ the environment generates $(s_{t+1}, r_{t+1})$.

> *Note - some authors use the convention that the agent's action causes the timestep transition, i.e. the trajectory looks like $s_0, a_0, \textcolor{red}{r_0}, s_1, a_1, \textcolor{red}{r_1}, \ldots$. We're following the convention in Sutton & Barto here, but it's important to be aware of this possible notation difference when reading other sources.*

The agent chooses actions using a policy $\pi$, which we can think of as either a deterministic function $a = \pi(s)$ from states to actions, or more generally a stochastic function from which actions are sampled, $a \sim \pi(\cdot | s)$.

Implicitly, we have assumed that the agent need only be Markovian as well (i.e. the action depends on the current state, not past states). Do you think this this a reasonable assumption? You should think about this before reading the dropdown.

<details>
<summary>Answer / discussion</summary>

In most environments, this may well be a reasonable assumption. For example, in a game of tic-tac-toe, knowledge of the current state of the board is sufficient to determine the optimal move, how the board got to that state is irrelevant.

There are more complex environments where a Markovian assumption wouldn't work. One classic example is a game of poker, since past player behaviour (betting patterns, bluffing, etc) might change what the optimal strategy is at a given point in time, even if the resulting game state is the same.

Of course, a non-Markovian game can be turned Markovian by redefining the state space to include relevant historical information, but this is splitting hairs (and it also blows up the state space).
</details>

### Value function

The goal of the agent is to choose a policy that maximizes the **expected discounted return** $G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \ldots$, the expected sum of future discounted rewards it would expect to obtain by following its currently chosen **policy** $\pi$. We call the expected discounted return from a state $s$ following policy $\pi$ the **value function** $V_{\pi}(s)$, defined as
$$
\begin{align*}
V_{\pi}(s) &= \mathbb{E}_{\pi} \left[ G_t \mid s_t = s \right] \\
&= \mathbb{E}_{\pi} \left[ \sum_{i=0}^\infty \gamma^{i} r_{i + t+1} \Bigg| s_t = s \right] \\
&= \mathbb{E}_{\pi} \left[ r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \ldots \mid s_t = s \right]
\end{align*}
$$
where the expectation is with respect to sampling actions from $\pi$, and (implicitly) sampling states and rewards from $T$ and $R$.

Here are a few divergences into the exact nature of the reward function. You can just read the TLDRs and move on if you're not super interested in the deeper mathematics.

<details>
<summary>Divergence #1: why do we discount? <i>TLDR - not discounting leads to possibly weird behaviour with infinite non-converging sums</i></summary>

We would like a way to signal to the agent that reward now is better than reward later.
If we didn't discount, the sum $\sum_{i=t}^\infty r_i$ may diverge.
This leads to strange behavior, as we can't meaningfully compare the returns for sequences of rewards that diverge. Trying to sum the sequence $1,1,1,\ldots$ or $2,2,2,2\ldots$ or even $2, -1, 2 , -1, \ldots$ is a problem since they all diverge to positive infinity. An agent with a policy that leads to infinite expected return might become lazy, as waiting around doing nothing for a thousand years, and then actually optimally leads to the same return (infinite) as playing optimal from the first time step.

Worse still, we can have reward sequences for which the sum may never approach any finite value, nor diverge to $\pm \infty$ (like summing $-1, 1, -1 ,1 , \ldots$).
We could otherwise patch this by requiring that the agent has a finite number of interactions with the environment (this is often true, and is called an **episodic** environment that we will see later) or restrict ourselves to environments for which the expected return for the optimal policy is finite, but these can often be undesirable constraints to place.

</details>

<details>
<summary>Divergence #2: why do we discount geometrically? <i>TLDR - geometric discounting is "time consistent", since the nature of the problem looks the same from each timestep.</i></summary>

In general, we could consider a more general discount function $\Gamma : \mathbb{N} \to [0,1)$ and define the discounted return as $\sum_{i=t}^\infty \Gamma(i) r_i$. The geometric discount $\Gamma(i) = \gamma^i$ is commonly used, but other discounts include the hyperbolic discount $\Gamma(i) = \frac{1}{1 + iD}$, where $D>0$ is a hyperparameter. (Humans are [often said](https://chris-said.io/2018/02/04/hyperbolic-discounting/) to act as if they use a hyperbolic discount.) Other than  being mathematically convenient to work with (as the sum of geometric discounts has an elegant closed form expression $\sum_{i=0}^\infty \gamma^i  = \frac{1}{1-\gamma}$), geometric is preferred as it is *time consistent*, that is, the discount from one time step to the next remains constant.
Rewards at timestep $t+1$ are always worth a factor of $\gamma$ less than rewards on timestep $t$

$$
\frac{\Gamma(t+1)}{\Gamma(t)} = \frac{\gamma^{t+1}}{\gamma^t} = \gamma
$$

whereas for the hyperbolic reward, the amount discounted from one step to the next
is a function of the timestep itself, and decays to 1 (no discount)

$$
\frac{\Gamma(t+1)}{\Gamma(t)} =  \frac{1+tD}{1+(t+1)D} \to 1
$$

so very little discounting is done once rewards are far away enough in the future.

Hence, using geometric discount lends a certain symmetry to the problem. If after $N$ actions the agent ends up in exactly the same state as it was in before, then there will be less total value remaining (because of the decay rate), but the optimal policy won't change (because the rewards the agent gets if it takes a series of actions will just be a scaled-down version of the rewards it would have gotten for those actions $N$ steps ago).

</details>


<details>
<summary> Divergence #3: why do both sides not depend on t? 
<i> TLDR - value function is time invariant. </i> </summary>

The value function does not have explicit time dependence on the left-hand side because the 
underlying process is **stationary** under policy $\pi$ (that is, the environment dynamics $T$ and $R$, and the policy $\pi$, all depend on what state you're in or what action you took, but not when it was done.)
Because of these assumptions, the distribution of future rewards depends only on the **current state** $s$, not on the absolute time $t$. By analogy, the best move in a game of chess depends entirely on the current board state, and not when during the game it happened. If we returned to the same board position twice in a game, the best move from that point is the same as what it was last time it was encountered.

</details>

### Bellman equation

We can write the value function in the following recursive manner, called the **Bellman equation**:

$$
V_\pi(s) = \sum_a \pi(a | s) \sum_{s'} T(s' | s, a) \bigg( R(s, a, s') + \gamma V_\pi(s') \bigg)
$$

(Optional) Try to prove this for yourself!

<details>
<summary>Hint</summary>

Start by writing the expectation as a sum over all possible actions $a$ that can be taken from state $s$. Then, try to separate $r_{t+1}$ and the later reward terms inside the sum.
</details>

<details>
<summary>Answer</summary>

Write this as a sum over actions according to the policy $\pi$:

$$
V_\pi(s) = \sum_a \pi(a \mid s) \; \mathbb{E}_\pi[G_t \mid s_t = s, a_t = a]
$$

Separate out the first reward $r_{t+1}$:

$$
V_\pi(s) = \sum_a \pi(a \mid s) \; \mathbb{E}_\pi \Big[ r_{t+1} + \gamma G_{t+1} \;\mid\; s_t = s, a_t = a \Big]
$$

Expand over next states $s'$ using the transition probability $T(s' \mid s, a)$ and reward function $R(s, a, s')$:

$$
V_\pi(s) = \sum_a \pi(a \mid s) \sum_{s'} T(s' \mid s, a) \Big[ R(s, a, s') + \gamma \; \mathbb{E}_\pi[G_{t+1} \mid s_{t+1} = s'] \Big]
$$

Recognize that $\mathbb{E}_\pi[G_{t+1} \mid s_{t+1} = s'] = V_\pi(s')$, giving the **Bellman equation**:

$$
V_\pi(s) = \sum_a \pi(a \mid s) \sum_{s'} T(s' \mid s, a) \big[ R(s, a, s') + \gamma V_\pi(s') \big]
$$

How should we interpret this? This formula tells us that the total value from present time can be written as sum of next-timestep rewards and value terms discounted by a factor of $\gamma$. Recall earlier in our discussion of **geometric** vs **hyperbolic** discounting, we argued that geometric discounting has a symmetry through time, because of the constant discount factor. This is exactly what this formula shows, just on the scale of a single step. If we had used a non-geometric discount, we wouldn't be able to get this nice recursive relationship (try and see why!)

</details>

The Bellman equation can be thought of as <i>"(value of following policy $\pi$ at current state) = (value of next reward, which is determined from following $\pi$) + (discounted value at next state if we continue following $\pi$)"</i>.

We can also define the **action-value function**, or **Q-value** of state $s$ and action $a$ following policy $\pi$:

$$
Q_\pi(s,a) = \mathbb{E}\left[ \sum_{i=0}^\infty \gamma^i r_{t+i+1} \Bigg| s_t=s, a_t=a   \right]
$$

which can be written recursively much like the value function can:

$$
Q_\pi(s,a) = \sum_{s'} T(s' \mid s, a) \bigg( R(s, a, s') + \gamma \sum_{a'} \pi(a' \mid s') Q_\pi(s', a') \bigg)
$$

This equation can be thought of as <i>"(value of choosing particular action $a$ in current state $s$, then afterwards following $\pi$) = (value of next reward, which is determined from action $a$) + (discounted value at next state if we continue following $\pi$)"</i>. In other words it's conceptually the same equation as before but rewritten in terms of $Q$, conditionining on what the next action is before we go back to following $\pi$.

<details>
<summary>Question - what do you think is the formula relating V and Q to each other?</summary>

$V_\pi(s)$ is the value at a given state according to policy $\pi$, but this must be the average of all the values of taking action $a$ at this state, weighted by the probability that they're taken (under $\pi$). So we have:

$$
V_\pi(s) = \sum_a \pi(a \mid s) Q_\pi(s, a)
$$

Substituting this into the Bellman equation for the Q-value, we get:

$$
Q_\pi(s,a) = \sum_{s'} T(s' \mid s, a) \big[ R(s, a, s') + \gamma V_\pi(s') \big]
$$

</details>

### Optimal policies

* Two policies $\pi_1$ and $\pi_2$ are **equivalent** if $V_{\pi_1}(s) = V_{\pi_2}(s)$ for all states $s$. 
* A policy $\pi_1$ is **better** than, or **pareto dominates**, $\pi_2$ (denoted $\pi_1 \geq \pi_2$) if $V_{\pi_1}(s) \geq V_{\pi_2}(s)$ for all states $s$.

This gives a **partial order** on policies, as it may be the case that two policies are not equivalent, nor is one better than the other ($\pi_1$ might have higher value than $\pi_2$ in some states, and lower value in others).

A policy is **optimal** if it is better than all other policies. Optimal policies $\pi_1^*, \pi_2^*, \ldots$ are denoted with an asterisk to distinguish them from non-optimal policies. Note an optimal policy may not necessarily be unique, but the optimal value function $V^*(s) := \sup_\pi V_{\pi}(s)$ is unique by definition, and is achieved by at least one policy (exists $\pi^*$ such that $V^*= V_{\pi^*}$).

Theorem: An optimal deterministic policy exists for any MDP with $\gamma < 1$.

<details>
<summary>Proof (Optional) <i> TLDR - Define a metric space of value functions, and show 
that the Bellman operator is a contraction mapping on it. </i> 
</summary>

We prove that both the optimal value function is unique, and show by construction the existance of an optimal policy.

Define the **Bellman optimality operator** $\mathcal{B}$ acting on a value function $V : \mathcal{S} \to \mathbb{R}$ as:

$$
(\mathcal{B} V)(s) = \max_{a \in \mathcal{A}} \sum_{s' \in \mathcal{S}} T(s' \mid s,a) \Big[ R(s,a,s') + \gamma V(s') \Big]
$$

- $\mathcal{B}$ maps value functions to value functions.
- $\mathcal{B}$ is a **contraction mapping** in the sup-norm with the contraction factor 
equal to the discount factor $\gamma < 1$:

$$
\|\mathcal{B}V_1 - \mathcal{B}V_2\|_\infty \le \gamma \|V_1 - V_2\|_\infty
$$

for any two value functions $V_1$ and $V_2$, and where the sup-norm is defined as:

$$
\|V\|_\infty = \max_{s \in \mathcal{S}} |V(s)|
$$
and the difference between two value functions is defined in the obvious way:
$$
(V_1 - V_2)(s) = V_1(s) - V_2(s)
$$

Because $\mathcal{B}$ is a contraction on the complete metric space of bounded functions $V: \mathcal{S} \to \mathbb{R}$, it has a **unique fixed point** $V^*$ such that:

$$
\mathcal{B} V^* = V^*
$$

The Bellman operator only improves value functions, in the sense that $(\mathcal{B}V)(s) \ge V(s)$ for all states $s$, as taking the max over actions can only increase or leave unchanged the value.  

This means that $V \leq \mathcal{B}V \leq \mathcal{B}^2 V \leq \ldots$, and since regardless of the choice of initial value function $V$ we always converge to the unique fixed point $V^*$, this makes $V^*(s) \geq V(s)$ for all states $s$, and any value function $V$.


Define a **greedy policy** w.r.t. $V^*$ as:

$$
\pi^*(s) \in \arg\max_{a \in \mathcal{A}} \sum_{s' \in \mathcal{S}} T(s' \mid s,a) \Big[ R(s,a,s') + \gamma V^*(s') \Big]
$$

- By definition, this policy achieves $V^*$ for all states:

$$
V_{\pi^*}(s) = V^*(s), \quad \forall s \in \mathcal{S}
$$

Note it may not be unique, as there may be multiple actions that achieve the maximum value.

</details>

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "2.1.9"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part1_intro_to_rl.solutions import Environment, policy_eval_numerical, policy_eval_exact, policy_improvement, find_optimal_policy, Cheater, EpsilonGreedy, TD_LambdaConfig, RandomAgent


### Exercise - Implement reward averaging
> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 20-30 minutes on this exercise.
> ```

You should now complete the methods for the `RewardAveraging` agent, which applies the reward averaging algorithm as detailed in Sutton and Barto section 2.4, "Incremental Implementation":

* You should fill in `observe` to keep track of the number of times $n$ each arm has been pushed, as well as the value $Q_n$ for each arm.
    * Gotcha - in S & B notation, $n$ is the number of times this particular arm has been pulled, not the total number of actions taken!
    * We recommend defining arrays `N` and `Q`, each of length `num_arms`, to keep track of all these values.
* You should fill in `get_action` with an epsilon-greedy method: taking a random action with probability `epsilon`, and taking the best action based on the current value of $Q$ with probability `1-epsilon` (see Sutton & Barto).
* You should fill in `reset` to call the reset method from the parent class, *and* make sure that the tracked values of $(n, Q_n)$ are set back to appropriate values at the start of each episode.
    * Note, the `reset` method is also called before the very first run, so you don't need to define `N` and `Q` in the init method.
    * The `Q` values should be initialized according to the optimism value of this agent.
    * Ensure the `Q` values are stored as float (and not numpy's default, integer)

<details>
<summary>Hint - average reward formula</summary>

$$Q_k = Q_{k-1} + \frac{1}{k}[R_k - Q_{k-1}]$$

Where $k$ is the number of times the action has been taken, $R_k$ is the reward from the kth time the action was taken, and $Q_{k-1}$ is the average reward from the previous times this action was taken (this notation departs slightly from the S&B notation, but may be more helpful for our implementation).

**Important - $k$ is not the total number of timesteps, it's the total number of times you've taken this particular action.**
</details>

We've given you a function for plotting multiple agents' reward trajectories on the same graph, with an optional moving average parameter to make the graph smoother.

In [ ]:
class RewardAveraging(Agent):
    def __init__(self, num_arms: int, seed: int, epsilon: float, optimism: float):
        self.epsilon = epsilon
        self.optimism = optimism
        super().__init__(num_arms, seed)

    def get_action(self):
        raise NotImplementedError("Implement the get_action method for the RewardAveraging class.")

    def observe(self, action, reward, info):
        raise NotImplementedError("Implement the observe method for the RewardAveraging class.")

    def reset(self, seed: int):
        raise NotImplementedError("Implement the reset method for the RewardAveraging class.")

    def __repr__(self):
        # For the legend, when plotting
        return f"RewardAveraging(eps={self.epsilon}, optimism={self.optimism})"


num_arms = 10
stationary = True
names = []
all_rewards = []
env = gym.make("ArmedBanditTestbed-v0", num_arms=num_arms, stationary=stationary)

for optimism in [0, 5]:
    agent = RewardAveraging(num_arms, 0, epsilon=0.01, optimism=optimism)
    (rewards, num_correct) = run_agent(env, agent, n_runs=N_RUNS, base_seed=1)
    all_rewards.append(rewards)
    names.append(str(agent))
    print(agent)
    print(f" -> Frequency of correct arm: {num_correct.mean():.4f}")
    print(f" -> Average reward: {rewards.mean():.4f}")

utils.plot_rewards(all_rewards, names, moving_avg_window=15)

<details>
<summary>Question - can you interpret these results?</summary>

At the very start, the more optimistic agent performs worse, because it explores more and exploits less (here you will see it at the very beginning if you zoom in). Its estimates are wildly over-optimistic, so even if it finds a good arm, its Q-value for that arm will decrease. On the other hand, if the realistic agent finds a good arm early on, it'll probably return to exploit it.

However, the optimistic agent eventually outperforms the realistic agent, because its increased exploration means it's more likely to converge on the best arm.
</details>

<details>
<summary>Question - how do you think these results would change if epsilon was decreased for both agents?</summary>

You should expect the optimistic agent to outperform the realistic agent even more. The smaller epsilon is, the more necessary optimism is (because without it the agent won't explore enough).
</details>


<details><summary>Solution</summary>

```python
class RewardAveraging(Agent):
    def __init__(self, num_arms: int, seed: int, epsilon: float, optimism: float):
        self.epsilon = epsilon
        self.optimism = optimism
        super().__init__(num_arms, seed)

    def get_action(self):
        if self.rng.random() < self.epsilon:
            return self.rng.integers(low=0, high=self.num_arms).item()
        else:
            return np.argmax(self.Q)

    def observe(self, action, reward, info):
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]

    def reset(self, seed: int):
        super().reset(seed)
        self.N = np.zeros(self.num_arms)
        self.Q = np.full(self.num_arms, self.optimism, dtype=float)

    def __repr__(self):
        # For the legend, when plotting
        return f"RewardAveraging(eps={self.epsilon}, optimism={self.optimism})"
```
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
